# Real Estate Recommender — Surprise SVD
Applies matrix factorization (SVD) via the `scikit-surprise` library to the implicit like data.

**Note:** All ratings are identical (implicit like = 1), so RMSE/MAE will be trivially zero.
This section is kept as a sanity-check baseline; use the `implicit` ALS or LightFM notebooks for meaningful ranking metrics.

**Requires:** `pip install scikit-surprise`

In [1]:
import numpy as np
import pandas as pd

try:
    import surprise
except ImportError:
    surprise = None
    print("scikit-surprise not installed. Run: pip install scikit-surprise")

scikit-surprise not installed. Run: pip install scikit-surprise


In [2]:
# Load datasets
users_df = pd.read_csv('data-refined/users.csv')
listings_df = pd.read_csv('data-refined/cleaned_listings.csv')
user_likes_df = pd.read_csv('data-refined/user_likes.csv')

print(f"Users: {users_df.shape[0]} | Listings: {listings_df.shape[0]} | Likes: {user_likes_df.shape[0]}")

Users: 200 | Listings: 26648 | Likes: 26546


## Surprise SVD — dataset setup

In [3]:
# Convert interaction DataFrame to a Surprise Dataset
if surprise is None:
    print("scikit-surprise not installed — skipping SVD section. Install via: pip install scikit-surprise")
else:
    from surprise import Dataset, Reader, SVD, accuracy
    from surprise.model_selection import train_test_split as surprise_train_test_split
    from surprise.model_selection import cross_validate as surprise_cross_validate
    from surprise.model_selection import KFold

    user_col = 'user_id'
    item_col = 'listing_id'

    # Implicit likes have no explicit rating column — assign 1.0
    df_rated = user_likes_df.copy()
    df_rated['rating'] = 1.0

    rating_min = df_rated['rating'].min()
    rating_max = df_rated['rating'].max()
    reader = Reader(rating_scale=(rating_min, rating_max))

    surprise_df = df_rated[[user_col, item_col, 'rating']].rename(
        columns={user_col: 'userID', item_col: 'itemID'}
    )
    data_surprise = Dataset.load_from_df(surprise_df[['userID', 'itemID', 'rating']], reader)

    trainset = data_surprise.build_full_trainset()
    print(f"Surprise trainset — users: {trainset.n_users}, items: {trainset.n_items}, ratings: {trainset.n_ratings}")
    print("All ratings are identical (implicit likes). RMSE/MAE will be trivially zero; use ranking metrics instead.")

    trainset_s, testset_s = surprise_train_test_split(data_surprise, test_size=0.2, random_state=42)

scikit-surprise not installed — skipping SVD section. Install via: pip install scikit-surprise


## 5-fold cross-validation

In [4]:
# 5-fold cross-validation on implicit like data using Surprise SVD
if surprise is None:
    print("scikit-surprise not installed — skipping cross-validation.")
else:
    if rating_min == rating_max:
        print('All interactions share the same rating (implicit like = 1). RMSE/MAE will be low by construction.')

    kf = KFold(n_splits=5, random_state=42, shuffle=True)
    algo = SVD(random_state=42)

    rmse_scores, mae_scores = [], []
    for fold, (trainset_cv, testset_cv) in enumerate(kf.split(data_surprise), start=1):
        algo.fit(trainset_cv)
        preds = algo.test(testset_cv)
        rmse = accuracy.rmse(preds, verbose=False)
        mae_val = accuracy.mae(preds, verbose=False)
        rmse_scores.append(rmse)
        mae_scores.append(mae_val)
        print(f'Fold {fold}: RMSE={rmse:.4f}, MAE={mae_val:.4f}')

    print()
    print(f'Mean RMSE: {np.mean(rmse_scores):.4f} ± {np.std(rmse_scores):.4f}')
    print(f'Mean MAE:  {np.mean(mae_scores):.4f} ± {np.std(mae_scores):.4f}')

scikit-surprise not installed — skipping cross-validation.
